In [4]:
import pandas as pd 
from matplotlib import pyplot as plt
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 
import spacy

# Main 
## Parteien Thesen und Texte

In [5]:
# Einlesen der Daten 
all_parties_df = pd.read_parquet("all_parties_text_combined.parquet")
wahlomat_theses_df= pd.read_parquet("wahlomat_thesen_positionen.parquet")

In [6]:
# Entfernen der Stopwords und Satzzeichen, Zahlen sollen erhalten bleiben, da potentiell weiterer Nutzen
def remove_stopwords(text): 
    if pd.isna(text) or text.strip() == "":  
        return ""

    tokens = [w for w in word_tokenize(text.lower())
                if w.isalnum() ]

    no_stops = [t for t in tokens
                    if t not in stopwords.words('german')]
    return " ".join(no_stops)

all_parties_df["Text_no_stopwords"] = all_parties_df["Text"].apply(remove_stopwords)
all_parties_df["These_no_stopwords"] = all_parties_df["These"].apply(remove_stopwords)

In [7]:
# Lemmatisieren der Texte mit Spacy
def lemmatize_words(text): 
    text = text.lower()
    doc = nlp(text)
    lemmatized = [token.lemma_ for token in doc]
    return " ".join(lemmatized)

nlp = spacy.load("de_core_news_md")
all_parties_df["Text_lemmatized"] = all_parties_df["Text_no_stopwords"].apply(lemmatize_words)
all_parties_df["These_lemmatized"] = all_parties_df["These_no_stopwords"].apply(lemmatize_words)

In [8]:
# Entfernen von Duplikaten in Thesen
def remove_duplicate_words(text):
    if pd.isna(text) or text.strip() == "":
        return "", 0 

    words = text.split()
    unique_words = list(dict.fromkeys(words)) 
    removed_count = len(words) - len(unique_words)
    return " ".join(unique_words), removed_count

all_parties_df["These_unique_words"], all_parties_df["Removed_Word_Count"] = zip(
    *all_parties_df["These_lemmatized"].apply(remove_duplicate_words)
)

In [9]:
all_parties_df = all_parties_df.drop(columns=["Removed_Word_Count"])

In [10]:
file_name = 'all_parties_text_combined_prep.parquet' 
all_parties_df.to_parquet(file_name, index=False)

## Wahlomat Thesen

In [11]:
# Entfernen der Stopwords und Satzzeichen, Zahlen sollen erhalten bleiben, da potentiell weiterer Nutzen
def remove_stopwords(text): 
    if pd.isna(text) or text.strip() == "":  
        return ""

    tokens = [w for w in word_tokenize(text.lower())
                if w.isalnum() ]

    no_stops = [t for t in tokens
                    if t not in stopwords.words('german')]
    return " ".join(no_stops)

wahlomat_theses_df["These_no_stopwords"] = wahlomat_theses_df["These"].apply(remove_stopwords)


In [12]:
# Lemmatisieren der Texte mit Spacy
def lemmatize_words(text): 
    text = text.lower()
    doc = nlp(text)
    lemmatized = [token.lemma_ for token in doc]
    return " ".join(lemmatized)

nlp = spacy.load("de_core_news_md")
wahlomat_theses_df["These_lemmatized"] = wahlomat_theses_df["These_no_stopwords"].apply(lemmatize_words)


In [13]:
# Entfernen von Duplikaten in Thesen
def remove_duplicate_words(text):
    if pd.isna(text) or text.strip() == "":
        return "", 0 

    words = text.split()
    unique_words = list(dict.fromkeys(words)) 
    removed_count = len(words) - len(unique_words)
    return " ".join(unique_words), removed_count

wahlomat_theses_df["These_unique_words"], wahlomat_theses_df["Removed_Word_Count"] = zip(
    *wahlomat_theses_df["These_lemmatized"].apply(remove_duplicate_words)
)

In [14]:
wahlomat_theses_df = wahlomat_theses_df.drop(columns=["Removed_Word_Count"])

In [15]:
# Spalten neu anordnen, damit die neuen Spalten hinter der ersten Spalte eingefügt werden
cols = ['These', 'These_no_stopwords', 'These_lemmatized', 'These_unique_words'] + [col for col in wahlomat_theses_df.columns if col not in ['These', 'These_no_stopwords', 'These_lemmatized', 'These_unique_words']]
wahlomat_theses_df = wahlomat_theses_df[cols]

# Speichern des DataFrames im Parquet-Format
file_name = 'wahlomat_thesen_prep.parquet' 
wahlomat_theses_df.to_parquet(file_name, index=False)

In [16]:
display(wahlomat_theses_df)

Partei,These,These_no_stopwords,These_lemmatized,These_unique_words,CDU / CSU,GRÜNE,SPD,AfD,Die Linke,FDP
0,Alle Beschäftigten sollen bereits nach 40 Beit...,beschäftigten sollen bereits 40 beitragsjahren...,beschäftigter sollen bereits 40 beitragsjahr a...,beschäftigter sollen bereits 40 beitragsjahr a...,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu
1,Alle Bürgerinnen und Bürger sollen in gesetzli...,bürgerinnen bürger sollen gesetzlichen kranken...,bürgerinn Bürger sollen gesetzlich Krankenkass...,bürgerinn Bürger sollen gesetzlich Krankenkass...,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu
2,An Bahnhöfen soll die Bundespolizei Software z...,bahnhöfen bundespolizei software automatisiert...,bahnhöfen Bundespolizei Software automatisiert...,bahnhöfen Bundespolizei Software automatisiert...,stimme zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu,stimme nicht zu
3,Asylsuchende sollen in Deutschland sofort nach...,asylsuchende sollen deutschland sofort antrags...,asylsuchender sollen Deutschland sofort Antrag...,asylsuchender sollen Deutschland sofort Antrag...,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,neutral
4,"Asylsuchende, die über einen anderen EU-Staat ...",asylsuchende eingereist sollen deutschen grenz...,asylsuchender einreisen sollen deutsch Grenze ...,asylsuchender einreisen sollen deutsch Grenze ...,stimme zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu,stimme zu
5,Auf allen Autobahnen soll ein generelles Tempo...,autobahnen generelles tempolimit gelten,autobahn Generelle tempolimit gelten,autobahn Generelle tempolimit gelten,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu
6,Aus Deutschland sollen weiterhin Rüstungsgüter...,deutschland sollen weiterhin rüstungsgüter isr...,Deutschland sollen weiterhin rüstungsgüter Isr...,Deutschland sollen weiterhin rüstungsgüter Isr...,stimme zu,stimme zu,stimme zu,neutral,stimme nicht zu,stimme zu
7,Bei Neuvermietungen sollen die Mietpreise weit...,neuvermietungen sollen mietpreise weiterhin ge...,Neuvermietung sollen mietpreise weiterhin gese...,Neuvermietung sollen mietpreise weiterhin gese...,stimme zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu
8,Bei der Besteuerung von Einkommen soll der Spi...,besteuerung einkommen spitzensteuersatz angehoben,Besteuerung Einkommen Spitzensteuersatz anheben,Besteuerung Einkommen Spitzensteuersatz anheben,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu
9,Beim Ausbau der Verkehrsinfrastruktur soll die...,beim ausbau verkehrsinfrastruktur schiene vorr...,bei Ausbau verkehrsinfrastruktur Schiene Vorra...,bei Ausbau verkehrsinfrastruktur Schiene Vorra...,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu
